# Proyecto final: exploración del dataset de enfermedades foliares

En este cuadernillo se realizó una exploración inicial del *Plant Disease Recognition Dataset*. El propósito fue verificar la estructura de los datos, las clases disponibles, la cantidad de imágenes y algunas características visuales antes de aplicar preprocesamiento y modelos de clasificación.

El objetivo preliminar del proyecto es clasificar imágenes de hojas en tres condiciones: **sana** (*Healthy*), **mildiu polvoriento** (*Powdery*) y **roya** (*Rust*).

Fuente del dataset: [Kaggle - Plant disease recognition dataset](https://www.kaggle.com/datasets/rashikrahmanpritom/plant-disease-recognition-dataset).

## 1. Librerías y rutas

Se cargaron las librerías necesarias para leer las imágenes, organizar la información y construir las visualizaciones. La ruta se definió de forma relativa al cuadernillo para que el proyecto pueda ejecutarse sin cambios cuando se conserve la estructura de carpetas.

In [ ]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')

RUTA_PROYECTO = Path.cwd()
RUTA_DATASET = RUTA_PROYECTO / 'dataset_plant_disease'
CLASES = ['Healthy', 'Powdery', 'Rust']
PARTICIONES = ['Train', 'Validation', 'Test']

print(f'Ruta del dataset: {RUTA_DATASET}')
print(f'Dataset encontrado: {RUTA_DATASET.exists()}')

## 2. Estructura y cantidad de imágenes

El dataset ya incluye una partición para entrenamiento, validación y prueba. Cada partición contiene las mismas tres clases. Esta separación se conservará durante el proyecto para evitar que las imágenes de prueba influyan en el entrenamiento o en la elección de hiperparámetros.

In [ ]:
def obtener_archivos(carpeta):
    return sorted(list(carpeta.glob('*.jpg')) + list(carpeta.glob('*.jpeg')) + list(carpeta.glob('*.png')))

registros = []
for particion in PARTICIONES:
    for clase in CLASES:
        # La estructura original tiene carpetas duplicadas: Train/Train, Test/Test y Validation/Validation.
        carpeta_clase = RUTA_DATASET / particion / particion / clase
        archivos = obtener_archivos(carpeta_clase)
        registros.append({
            'Partición': particion,
            'Clase': clase,
            'Cantidad de imágenes': len(archivos),
            'Ruta': carpeta_clase
        })

df_conteo = pd.DataFrame(registros)
tabla_conteo = df_conteo.pivot(index='Clase', columns='Partición', values='Cantidad de imágenes')
tabla_conteo['Total'] = tabla_conteo.sum(axis=1)
tabla_conteo.loc['Total'] = tabla_conteo.sum(axis=0)
tabla_conteo

In [ ]:
orden_particiones = ['Train', 'Validation', 'Test']
fig, ax = plt.subplots(figsize=(9, 5))
for clase in CLASES:
    datos = (df_conteo[df_conteo['Clase'] == clase]
             .set_index('Partición')
             .reindex(orden_particiones)['Cantidad de imágenes'])
    ax.bar(np.arange(len(orden_particiones)) + CLASES.index(clase) * 0.25, datos, width=0.25, label=clase)

ax.set_xticks(np.arange(len(orden_particiones)) + 0.25)
ax.set_xticklabels(['Entrenamiento', 'Validación', 'Prueba'])
ax.set_ylabel('Cantidad de imágenes')
ax.set_title('Distribución de imágenes por partición y clase')
ax.legend(title='Clase')
plt.show()

## 3. Muestra visual de las clases

Se visualizaron ejemplos aleatorios de cada condición. Esta revisión permite confirmar que las etiquetas corresponden con la apariencia general de las hojas y anticipar posibles dificultades, como iluminación, fondos variados o lesiones de diferente tamaño.

In [ ]:
rng = np.random.default_rng(42)
fig, axes = plt.subplots(3, 3, figsize=(12, 11))

for fila, clase in enumerate(CLASES):
    carpeta = RUTA_DATASET / 'Train' / 'Train' / clase
    muestra = rng.choice(obtener_archivos(carpeta), size=3, replace=False)
    for columna, ruta_imagen in enumerate(muestra):
        with Image.open(ruta_imagen) as imagen:
            axes[fila, columna].imshow(imagen.convert('RGB'))
        axes[fila, columna].set_title(clase)
        axes[fila, columna].axis('off')

fig.suptitle('Ejemplos de imágenes del conjunto de entrenamiento', fontsize=15, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 4. Tamaño y formato de las imágenes

Se revisó el tamaño de las imágenes del conjunto de entrenamiento. Este análisis permite escoger un tamaño de entrada para los modelos y cuantificar la reducción de información que se realizará al redimensionar las imágenes.

In [ ]:
registros_tamano = []
for clase in CLASES:
    for ruta_imagen in obtener_archivos(RUTA_DATASET / 'Train' / 'Train' / clase):
        with Image.open(ruta_imagen) as imagen:
            ancho, alto = imagen.size
            registros_tamano.append({'Clase': clase, 'Ancho': ancho, 'Alto': alto, 'Modo': imagen.mode})

df_tamanos = pd.DataFrame(registros_tamano)
resumen_tamanos = df_tamanos.groupby('Clase')[['Ancho', 'Alto']].agg(['min', 'median', 'max'])
resumen_tamanos

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for clase in CLASES:
    datos = df_tamanos[df_tamanos['Clase'] == clase]
    ax.scatter(datos['Ancho'], datos['Alto'], alpha=0.55, label=clase)

ax.set_xlabel('Ancho (píxeles)')
ax.set_ylabel('Alto (píxeles)')
ax.set_title('Dimensiones originales de las imágenes de entrenamiento')
ax.legend(title='Clase')
plt.show()

print('Formatos encontrados:')
display(df_tamanos['Modo'].value_counts().rename_axis('Modo').to_frame('Cantidad'))

## 5. Conclusiones de la exploración inicial

- Se verificó que el problema corresponde a una **clasificación multiclase supervisada**, ya que cada imagen está asociada a una de tres etiquetas.
- Se conservarán las particiones originales de entrenamiento, validación y prueba.
- En los siguientes cuadernillos se evaluará el efecto de redimensionamiento, espacios de color, ecualización local mediante CLAHE y extracción de características LBP.
- Posteriormente se entrenará un modelo clásico basado en LBP y SVM, que se comparará con una CNN mediante transferencia de aprendizaje.
- Las predicciones se interpretarán como clasificación visual de las tres clases del dataset, no como un diagnóstico agrícola general.

## 6. Preprocesamiento: tamaño, escala de grises, LAB y CLAHE

Para el enfoque clásico basado en LBP se utilizarán imágenes en escala de grises y de tamaño fijo. La estandarización a 128 × 128 píxeles permite que todos los histogramas LBP tengan la misma longitud. Más adelante, para la CNN con transferencia de aprendizaje, se utilizará el tamaño de 224 × 224 píxeles requerido por la arquitectura seleccionada.

También se exploró el espacio de color **LAB**. En este espacio, el canal **L** representa la luminosidad o cantidad de luz; el canal **a** representa aproximadamente el eje verde--rojo y el canal **b** el eje azul--amarillo. Esta separación permite modificar el contraste de la luz sin alterar directamente los componentes de color.

Se aplicó **CLAHE** (*Contrast Limited Adaptive Histogram Equalization* o ecualización adaptativa de histograma con límite de contraste) únicamente sobre el canal L. CLAHE divide la imagen en pequeñas regiones, mejora el contraste dentro de cada región y luego une los resultados. El parámetro `clipLimit` limita cuánto puede crecer el contraste, con el fin de evitar que también se amplifique demasiado el ruido. Al reconstruir la imagen con los canales a y b originales, se conserva el color natural de la hoja. Frente a una ecualización global, CLAHE es útil cuando una parte de la hoja está en sombra y otra está iluminada. Como limitación, un `clipLimit` alto puede resaltar ruido, sombras o detalles poco relevantes.

In [ ]:
import cv2

TAMANO_LBP = (128, 128)
TAMANO_CNN = (224, 224)

def leer_imagen_rgb(ruta):
    """Lee una imagen mediante PIL y la entrega en formato RGB."""
    with Image.open(ruta) as imagen:
        return np.array(imagen.convert('RGB'))

def redimensionar_imagen(imagen_rgb, tamano):
    return cv2.resize(imagen_rgb, tamano, interpolation=cv2.INTER_AREA)

def clahe_en_luminosidad(imagen_rgb, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Aplica CLAHE al canal L y conserva los canales de color originales."""
    imagen_lab = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2LAB)
    canal_l, canal_a, canal_b = cv2.split(imagen_lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    canal_l_clahe = clahe.apply(canal_l)
    lab_clahe = cv2.merge([canal_l_clahe, canal_a, canal_b])
    rgb_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return canal_l, canal_l_clahe, rgb_clahe

ruta_ejemplo = next((RUTA_DATASET / 'Train' / 'Train' / 'Powdery').glob('*.jpg'))
imagen_original = leer_imagen_rgb(ruta_ejemplo)
imagen_128 = redimensionar_imagen(imagen_original, TAMANO_LBP)
imagen_gris = cv2.cvtColor(imagen_128, cv2.COLOR_RGB2GRAY)
canal_l, canal_l_clahe, imagen_clahe = clahe_en_luminosidad(imagen_128, clip_limit=2.0)

print(f'Imagen de ejemplo: {ruta_ejemplo.name}')
print(f'Tamaño original: {imagen_original.shape[1]} × {imagen_original.shape[0]} píxeles')
print(f'Tamaño para LBP: {TAMANO_LBP[0]} × {TAMANO_LBP[1]} píxeles')
print(f'Tamaño previsto para CNN: {TAMANO_CNN[0]} × {TAMANO_CNN[1]} píxeles')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(imagen_original)
axes[0, 0].set_title('Imagen original')
axes[0, 1].imshow(imagen_128)
axes[0, 1].set_title('Redimensionada a 128 × 128')
axes[0, 2].imshow(imagen_gris, cmap='gray')
axes[0, 2].set_title('Escala de grises')

axes[1, 0].imshow(canal_l, cmap='gray')
axes[1, 0].set_title('Canal L de LAB')
axes[1, 1].imshow(canal_l_clahe, cmap='gray')
axes[1, 1].set_title('Canal L después de CLAHE')
axes[1, 2].imshow(imagen_clahe)
axes[1, 2].set_title('Imagen LAB reconstruida con CLAHE')

for ax in axes.ravel():
    ax.axis('off')

fig.suptitle('Etapas de preprocesamiento en una hoja con mildiu polvoriento', fontsize=15, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(canal_l.ravel(), bins=256, range=(0, 256), alpha=0.65, label='Canal L original', color='gray')
ax.hist(canal_l_clahe.ravel(), bins=256, range=(0, 256), alpha=0.55, label='Canal L con CLAHE', color='tab:orange')
ax.set_xlabel('Intensidad de luminosidad')
ax.set_ylabel('Frecuencia de píxeles')
ax.set_title('Efecto de CLAHE sobre la luminosidad')
ax.legend()
plt.show()

### Interpretación

Se observó que CLAHE redistribuye los niveles de luminosidad de forma local. En consecuencia, pueden hacerse más visibles los detalles de textura asociados a manchas o zonas blanquecinas. El histograma permite ver este cambio: cuando los valores de luminosidad se distribuyen en un rango más amplio, es más fácil distinguir zonas claras y oscuras. Para LBP se comparará posteriormente si extraer las características desde la escala de grises original o desde la imagen con CLAHE genera una mejor clasificación.

## 7. Extracción de características mediante LBP

Local Binary Patterns (LBP) describe la textura local de una imagen. Para cada píxel central, se compara su intensidad con la de sus vecinos: se asigna 1 cuando el vecino tiene una intensidad igual o mayor y 0 en el caso contrario. La secuencia de ceros y unos forma un código que representa microtexturas como puntos, líneas, bordes y zonas homogéneas.

En este proyecto se utilizó inicialmente LBP uniforme con **P = 8** vecinos y **R = 1** píxel. P es el número de vecinos que se revisan alrededor del píxel central y R es la distancia desde el centro hasta esos vecinos. El método `uniform` agrupa patrones simples, es decir, aquellos que cambian pocas veces entre 0 y 1 al recorrer el vecindario; por ello se obtiene un descriptor más compacto y menos sensible a pequeñas variaciones. Con P = 8, el histograma contiene diez códigos posibles, de 0 a 9. Una barra alta indica que ese patrón local aparece con mayor frecuencia; no significa automáticamente que exista más textura.

In [ ]:
from skimage.feature import local_binary_pattern

P_INICIAL = 8
R_INICIAL = 1

def calcular_lbp(imagen_gris, puntos=P_INICIAL, radio=R_INICIAL):
    """Calcula LBP uniforme para una imagen en escala de grises."""
    return local_binary_pattern(imagen_gris, P=puntos, R=radio, method='uniform')

def histograma_lbp(imagen_lbp, puntos=P_INICIAL):
    """Obtiene un histograma normalizado de LBP uniforme."""
    limites = np.arange(0, puntos + 3)
    histograma, _ = np.histogram(imagen_lbp.ravel(), bins=limites, range=(0, puntos + 2), density=True)
    return histograma

muestras_por_clase = {}
for clase in CLASES:
    ruta = next((RUTA_DATASET / 'Train' / 'Train' / clase).glob('*.jpg'))
    rgb = redimensionar_imagen(leer_imagen_rgb(ruta), TAMANO_LBP)
    gris = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    lbp = calcular_lbp(gris)
    muestras_por_clase[clase] = {'ruta': ruta, 'gris': gris, 'lbp': lbp, 'histograma': histograma_lbp(lbp)}

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12))

for fila, clase in enumerate(CLASES):
    datos = muestras_por_clase[clase]
    axes[fila, 0].imshow(datos['gris'], cmap='gray')
    axes[fila, 0].set_title(f'{clase}: escala de grises')
    axes[fila, 1].imshow(datos['lbp'], cmap='gray')
    axes[fila, 1].set_title(f'{clase}: imagen LBP')
    ejes_x = np.arange(len(datos['histograma']))
    axes[fila, 2].bar(ejes_x, datos['histograma'], color='tab:green')
    axes[fila, 2].set_xticks(ejes_x)
    axes[fila, 2].set_xlabel('Código LBP uniforme')
    axes[fila, 2].set_ylabel('Frecuencia normalizada')
    axes[fila, 2].set_title(f'{clase}: histograma LBP')

for ax in axes[:, :2].ravel():
    ax.axis('off')

fig.suptitle('Representación LBP de una muestra por clase (P = 8, R = 1)', fontsize=15, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

### Comparación de parámetros de LBP

Se compararon tres configuraciones. Al aumentar P y R se analiza una zona más amplia alrededor de cada píxel. Esto puede capturar patrones de mayor escala, pero también puede perder detalles finos. Para las manchas y texturas locales de las hojas, la configuración inicial P = 8 y R = 1 será la referencia.

In [ ]:
configuraciones_lbp = [(8, 1), (16, 2), (24, 3)]
imagen_comparacion = muestras_por_clase['Powdery']['gris']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for columna, (puntos, radio) in enumerate(configuraciones_lbp):
    lbp = calcular_lbp(imagen_comparacion, puntos=puntos, radio=radio)
    hist = histograma_lbp(lbp, puntos=puntos)
    axes[0, columna].imshow(lbp, cmap='gray')
    axes[0, columna].set_title(f'LBP: P={puntos}, R={radio}')
    axes[0, columna].axis('off')
    axes[1, columna].bar(np.arange(len(hist)), hist, color='tab:orange')
    axes[1, columna].set_title(f'Histograma: P={puntos}, R={radio}')
    axes[1, columna].set_xlabel('Código LBP')
    axes[1, columna].set_ylabel('Frecuencia normalizada')

fig.suptitle('Efecto del tamaño del vecindario LBP en una hoja con mildiu', fontsize=15, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

### Decisión inicial

Se utilizará inicialmente LBP uniforme con P = 8 y R = 1 porque representa microtexturas con un vector compacto. Más adelante, se evaluará de manera cuantitativa si esta configuración, una configuración de mayor radio o la extracción sobre imágenes con CLAHE proporciona mejores resultados al clasificar las tres condiciones foliares mediante SVM.

## 8. Clasificación clásica: LBP + SVM

En este enfoque se extrajeron histogramas LBP en una cuadrícula de 4 × 4 bloques. Esta división conserva parte de la ubicación de las texturas en la hoja y evita que el histograma global pierda por completo la información espacial. Los histogramas concatenados se utilizaron como entrada de una Máquina de Vectores de Soporte (SVM) lineal.

Una SVM busca fronteras que separen los vectores de características de las clases. En este caso, cada hoja queda representada por 160 valores de textura y la SVM aprende a asociar esos valores con las clases Healthy, Powdery y Rust. Antes del entrenamiento se estandarizan las características para que ninguna tenga una influencia excesiva por su escala.

Los hiperparámetros se seleccionarán con el conjunto de validación. El parámetro C controla cuánto se penalizan los errores durante el ajuste: valores grandes intentan corregir más errores de entrenamiento, mientras que valores pequeños permiten una frontera más simple. El conjunto de prueba permanecerá reservado para la evaluación final del modelo seleccionado. Debido a que el modelo usa imágenes de 128 × 128, las fotografías JPEG se decodifican inicialmente a resolución reducida antes de la estandarización; esto disminuye el tiempo de cómputo sin cambiar el tamaño de entrada del modelo.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

BLOQUES_LBP = 4
USAR_CLAHE_LBP = True

def caracteristicas_lbp_espaciales(imagen_gris, puntos=P_INICIAL, radio=R_INICIAL, bloques=BLOQUES_LBP):
    """Concatena histogramas LBP uniformes de una cuadrícula regular."""
    imagen_lbp = calcular_lbp(imagen_gris, puntos=puntos, radio=radio)
    alto, ancho = imagen_lbp.shape
    caracteristicas = []

    for fila in range(bloques):
        for columna in range(bloques):
            y_inicial, y_final = fila * alto // bloques, (fila + 1) * alto // bloques
            x_inicial, x_final = columna * ancho // bloques, (columna + 1) * ancho // bloques
            bloque_lbp = imagen_lbp[y_inicial:y_final, x_inicial:x_final]
            caracteristicas.extend(histograma_lbp(bloque_lbp, puntos=puntos))

    return np.asarray(caracteristicas, dtype=np.float32)

def leer_imagen_reducida_para_modelo(ruta):
    """Lee JPEG a resolución reducida; posteriormente se estandariza a 128 × 128."""
    datos = np.fromfile(str(ruta), dtype=np.uint8)
    imagen_bgr = cv2.imdecode(datos, cv2.IMREAD_REDUCED_COLOR_8)
    if imagen_bgr is None:
        raise ValueError(f'No fue posible leer la imagen: {ruta}')
    return cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

def preparar_imagen_para_lbp(ruta, usar_clahe=USAR_CLAHE_LBP):
    imagen_rgb = redimensionar_imagen(leer_imagen_reducida_para_modelo(ruta), TAMANO_LBP)
    if usar_clahe:
        canal_l, canal_l_clahe, _ = clahe_en_luminosidad(imagen_rgb, clip_limit=2.0)
        return canal_l_clahe
    return cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2GRAY)

def cargar_particion_lbp(particion, usar_clahe=USAR_CLAHE_LBP):
    caracteristicas, etiquetas = [], []
    for etiqueta, clase in enumerate(CLASES):
        rutas = obtener_archivos(RUTA_DATASET / particion / particion / clase)
        for ruta in rutas:
            gris = preparar_imagen_para_lbp(ruta, usar_clahe=usar_clahe)
            caracteristicas.append(caracteristicas_lbp_espaciales(gris))
            etiquetas.append(etiqueta)
    return np.vstack(caracteristicas), np.asarray(etiquetas)

RUTA_CACHE_LBP = RUTA_PROYECTO / f'cache_lbp_P{P_INICIAL}_R{R_INICIAL}_B{BLOQUES_LBP}_clahe{USAR_CLAHE_LBP}.npz'

if RUTA_CACHE_LBP.exists():
    datos_lbp = np.load(RUTA_CACHE_LBP)
    X_entrenamiento, y_entrenamiento = datos_lbp['X_entrenamiento'], datos_lbp['y_entrenamiento']
    X_validacion, y_validacion = datos_lbp['X_validacion'], datos_lbp['y_validacion']
    X_prueba, y_prueba = datos_lbp['X_prueba'], datos_lbp['y_prueba']
    print('Características LBP cargadas desde caché.')
else:
    X_entrenamiento, y_entrenamiento = cargar_particion_lbp('Train')
    X_validacion, y_validacion = cargar_particion_lbp('Validation')
    X_prueba, y_prueba = cargar_particion_lbp('Test')
    np.savez_compressed(
        RUTA_CACHE_LBP,
        X_entrenamiento=X_entrenamiento, y_entrenamiento=y_entrenamiento,
        X_validacion=X_validacion, y_validacion=y_validacion,
        X_prueba=X_prueba, y_prueba=y_prueba
    )
    print(f'Características LBP guardadas en: {RUTA_CACHE_LBP.name}')

print(f'Características por imagen: {X_entrenamiento.shape[1]}')
print(f'Entrenamiento: {X_entrenamiento.shape}')
print(f'Validación: {X_validacion.shape}')
print(f'Prueba: {X_prueba.shape}')
print(f'CLAHE antes de LBP: {USAR_CLAHE_LBP}')

### Selección de hiperparámetros con validación

El parámetro **C** controla la penalización por errores durante el entrenamiento. Se evaluaron varios valores de C para una SVM lineal y se escogió el que obtuvo el mayor F1 macro en validación. Esta métrica da el mismo peso a las tres clases.

In [ ]:
candidatos_svm = [0.01, 0.1, 1, 10]

resultados_validacion = []

for valor_c in candidatos_svm:
    modelo = Pipeline([
        ('escalador', StandardScaler()),
        ('svm', LinearSVC(C=valor_c, dual='auto', max_iter=10000, random_state=42))
    ])
    modelo.fit(X_entrenamiento, y_entrenamiento)
    prediccion_validacion = modelo.predict(X_validacion)
    resultados_validacion.append({
        'C': valor_c,
        'Accuracy validación': accuracy_score(y_validacion, prediccion_validacion),
        'F1 macro validación': f1_score(y_validacion, prediccion_validacion, average='macro')
    })

df_validacion = pd.DataFrame(resultados_validacion).sort_values('F1 macro validación', ascending=False).reset_index(drop=True)
display(df_validacion.style.format({'Accuracy validación': '{:.3f}', 'F1 macro validación': '{:.3f}'}))

mejor_indice = df_validacion.index[0]
mejores_parametros = {'C': df_validacion.loc[mejor_indice, 'C']}
print(f"Mejor configuración según validación: C={mejores_parametros['C']}")

### Evaluación final en el conjunto de prueba

Después de seleccionar los hiperparámetros, se unieron entrenamiento y validación para ajustar el modelo final. El conjunto de prueba se utilizó únicamente en esta etapa para estimar el desempeño final del enfoque LBP + SVM.

In [ ]:
X_entrenamiento_final = np.vstack([X_entrenamiento, X_validacion])
y_entrenamiento_final = np.concatenate([y_entrenamiento, y_validacion])

modelo_lbp_svm = Pipeline([
    ('escalador', StandardScaler()),
    ('svm', LinearSVC(C=mejores_parametros['C'], dual='auto', max_iter=10000, random_state=42))
])
modelo_lbp_svm.fit(X_entrenamiento_final, y_entrenamiento_final)
prediccion_prueba = modelo_lbp_svm.predict(X_prueba)

accuracy_lbp_svm = accuracy_score(y_prueba, prediccion_prueba)
f1_lbp_svm = f1_score(y_prueba, prediccion_prueba, average='macro')

print(f'Accuracy en prueba: {accuracy_lbp_svm:.3f}')
print(f'F1 macro en prueba: {f1_lbp_svm:.3f}')
print('\nReporte de clasificación:')
print(classification_report(y_prueba, prediccion_prueba, target_names=CLASES, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_prueba, prediccion_prueba, display_labels=CLASES, cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Matriz de confusión: LBP + SVM')
plt.show()

### Interpretación inicial del enfoque clásico

El modelo LBP + SVM utiliza únicamente información de textura en escala de grises. Por esta razón, puede reconocer patrones relacionados con manchas o polvillo, pero no aprovecha directamente el color verde, blanco, naranja o café de las hojas. Esta limitación se tendrá en cuenta al comparar el modelo clásico con la CNN, la cual recibirá imágenes RGB y podrá aprender conjuntamente color, textura y forma.

## 9. CNN con transferencia de aprendizaje: MobileNetV2

Como segundo enfoque se utilizó MobileNetV2 preentrenada con ImageNet. Una red neuronal convolucional (CNN) aprende filtros que detectan patrones visuales; las primeras capas suelen reconocer bordes y colores, mientras que las capas más profundas combinan esos elementos en texturas y formas más complejas. MobileNetV2 ya fue entrenada con muchas imágenes, por lo que sus capas convolucionales identifican patrones generales útiles.

Estas capas se mantuvieron congeladas y se usaron como extractor de características. Congelar significa que sus pesos no se modifican durante este proyecto; así se reduce el tiempo de entrenamiento y el riesgo de sobreajuste, dado que el conjunto de hojas es relativamente pequeño. Sobre las 1280 características obtenidas se entrenó una capa densa de tres salidas, una por cada clase del proyecto. La salida con mayor probabilidad se toma como la clase predicha.

La arquitectura empleada fue:

```text
Imagen RGB 224 × 224
        ↓
MobileNetV2 preentrenada y congelada
        ↓
Global Average Pooling: vector de 1280 características
        ↓
Capa densa de 3 neuronas
        ↓
Softmax: Healthy / Powdery / Rust
```

Se empleó transferencia de aprendizaje porque el dataset es relativamente pequeño para entrenar una CNN desde cero y el entrenamiento se realiza únicamente con CPU.

In [ ]:
import copy
import os
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from torchvision import transforms

# Los pesos preentrenados se guardan dentro del proyecto para evitar rutas protegidas.
os.environ['TORCH_HOME'] = str(RUTA_PROYECTO / 'torch_cache')

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
DISPOSITIVO = torch.device('cpu')
BATCH_SIZE_CNN = 32
EPOCAS_MAXIMAS = 30
TASA_APRENDIZAJE = 1e-3

print(f'Dispositivo utilizado: {DISPOSITIVO}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
class DatasetHojas(Dataset):
    """Dataset que conserva las etiquetas originales y prepara imágenes RGB para MobileNetV2."""
    def __init__(self, particion, transformacion):
        self.rutas = []
        self.etiquetas = []
        self.transformacion = transformacion

        for etiqueta, clase in enumerate(CLASES):
            rutas_clase = obtener_archivos(RUTA_DATASET / particion / particion / clase)
            self.rutas.extend(rutas_clase)
            self.etiquetas.extend([etiqueta] * len(rutas_clase))

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, indice):
        # Se usa lectura reducida porque todas las imágenes se convertirán a 224 × 224.
        imagen_rgb = leer_imagen_reducida_para_modelo(self.rutas[indice])
        imagen_pil = Image.fromarray(imagen_rgb)
        return self.transformacion(imagen_pil), self.etiquetas[indice]

pesos_mobilenet = MobileNet_V2_Weights.DEFAULT
transformacion_cnn = pesos_mobilenet.transforms()

dataset_cnn_train = DatasetHojas('Train', transformacion_cnn)
dataset_cnn_val = DatasetHojas('Validation', transformacion_cnn)
dataset_cnn_test = DatasetHojas('Test', transformacion_cnn)

print(f'Imágenes para CNN - entrenamiento: {len(dataset_cnn_train)}')
print(f'Imágenes para CNN - validación: {len(dataset_cnn_val)}')
print(f'Imágenes para CNN - prueba: {len(dataset_cnn_test)}')

In [ ]:
# Se descargan los pesos la primera vez y posteriormente quedan en caché local.
modelo_base = mobilenet_v2(weights=pesos_mobilenet).to(DISPOSITIVO)
modelo_base.eval()
for parametro in modelo_base.parameters():
    parametro.requires_grad = False

def extraer_embeddings(dataset, batch_size=BATCH_SIZE_CNN):
    """Obtiene una representación de 1280 valores por imagen mediante MobileNetV2 congelada."""
    cargador = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    lista_embeddings, lista_etiquetas = [], []

    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            imagenes = imagenes.to(DISPOSITIVO)
            caracteristicas = modelo_base.features(imagenes)
            caracteristicas = nn.functional.adaptive_avg_pool2d(caracteristicas, (1, 1))
            caracteristicas = torch.flatten(caracteristicas, start_dim=1)
            lista_embeddings.append(caracteristicas.cpu())
            lista_etiquetas.append(etiquetas)

    return torch.cat(lista_embeddings), torch.cat(lista_etiquetas)

RUTA_CACHE_CNN = RUTA_PROYECTO / 'cache_embeddings_mobilenetv2.pt'

if RUTA_CACHE_CNN.exists():
    datos_cnn = torch.load(RUTA_CACHE_CNN, map_location='cpu', weights_only=True)
    emb_train, etq_train = datos_cnn['emb_train'], datos_cnn['etq_train']
    emb_val, etq_val = datos_cnn['emb_val'], datos_cnn['etq_val']
    emb_test, etq_test = datos_cnn['emb_test'], datos_cnn['etq_test']
    print('Embeddings de MobileNetV2 cargados desde caché.')
else:
    emb_train, etq_train = extraer_embeddings(dataset_cnn_train)
    emb_val, etq_val = extraer_embeddings(dataset_cnn_val)
    emb_test, etq_test = extraer_embeddings(dataset_cnn_test)
    torch.save({
        'emb_train': emb_train, 'etq_train': etq_train,
        'emb_val': emb_val, 'etq_val': etq_val,
        'emb_test': emb_test, 'etq_test': etq_test
    }, RUTA_CACHE_CNN)
    print(f'Embeddings de MobileNetV2 guardados en: {RUTA_CACHE_CNN.name}')

print(f'Forma de los embeddings de entrenamiento: {tuple(emb_train.shape)}')

### Entrenamiento de la capa de clasificación

La capa final se entrenó usando los embeddings obtenidos de MobileNetV2. Un embedding es un resumen numérico de la imagen: en este caso, un vector de 1280 valores que conserva patrones detectados por la CNN. Cada época corresponde a una pasada completa por las imágenes de entrenamiento. Se monitoreó la accuracy de validación y se conservaron los pesos de la época con mejor resultado. La detención temprana evita continuar el entrenamiento cuando el desempeño de validación deja de mejorar, reduciendo el riesgo de sobreajuste.

In [ ]:
class CabezaClasificacion(nn.Module):
    def __init__(self, entradas=1280, clases=3):
        super().__init__()
        self.clasificador = nn.Linear(entradas, clases)

    def forward(self, x):
        return self.clasificador(x)

cargador_entrenamiento_cnn = DataLoader(TensorDataset(emb_train, etq_train), batch_size=BATCH_SIZE_CNN, shuffle=True)
modelo_cnn = CabezaClasificacion().to(DISPOSITIVO)
perdida = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo_cnn.parameters(), lr=TASA_APRENDIZAJE)

historial_cnn = {'epoca': [], 'loss_entrenamiento': [], 'accuracy_entrenamiento': [], 'accuracy_validacion': []}
mejor_accuracy_val = -np.inf
mejor_estado = None
paciencia, sin_mejora = 6, 0

for epoca in range(1, EPOCAS_MAXIMAS + 1):
    modelo_cnn.train()
    perdida_acumulada, aciertos, total = 0.0, 0, 0

    for caracteristicas, etiquetas in cargador_entrenamiento_cnn:
        caracteristicas, etiquetas = caracteristicas.to(DISPOSITIVO), etiquetas.to(DISPOSITIVO)
        optimizador.zero_grad()
        logits = modelo_cnn(caracteristicas)
        valor_perdida = perdida(logits, etiquetas)
        valor_perdida.backward()
        optimizador.step()

        perdida_acumulada += valor_perdida.item() * etiquetas.size(0)
        aciertos += (logits.argmax(dim=1) == etiquetas).sum().item()
        total += etiquetas.size(0)

    modelo_cnn.eval()
    with torch.no_grad():
        pred_val = modelo_cnn(emb_val.to(DISPOSITIVO)).argmax(dim=1).cpu()
    accuracy_val = (pred_val == etq_val).float().mean().item()

    historial_cnn['epoca'].append(epoca)
    historial_cnn['loss_entrenamiento'].append(perdida_acumulada / total)
    historial_cnn['accuracy_entrenamiento'].append(aciertos / total)
    historial_cnn['accuracy_validacion'].append(accuracy_val)

    if accuracy_val > mejor_accuracy_val:
        mejor_accuracy_val = accuracy_val
        mejor_estado = copy.deepcopy(modelo_cnn.state_dict())
        sin_mejora = 0
    else:
        sin_mejora += 1

    if sin_mejora >= paciencia:
        print(f'Detención temprana en la época {epoca}.')
        break

modelo_cnn.load_state_dict(mejor_estado)
df_historial_cnn = pd.DataFrame(historial_cnn)
print(f'Mejor accuracy de validación: {mejor_accuracy_val:.3f}')
display(df_historial_cnn.tail())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_historial_cnn['epoca'], df_historial_cnn['loss_entrenamiento'], marker='o', color='tab:red')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Pérdida durante el entrenamiento')

axes[1].plot(df_historial_cnn['epoca'], df_historial_cnn['accuracy_entrenamiento'], marker='o', label='Entrenamiento')
axes[1].plot(df_historial_cnn['epoca'], df_historial_cnn['accuracy_validacion'], marker='o', label='Validación')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy de entrenamiento y validación')
axes[1].legend()
plt.tight_layout()
plt.show()

### Evaluación final de la CNN

El conjunto de prueba se evaluó con la misma partición reservada utilizada para LBP + SVM. Esto permite comparar ambos métodos de manera consistente.

In [ ]:
modelo_cnn.eval()
with torch.no_grad():
    prediccion_cnn = modelo_cnn(emb_test.to(DISPOSITIVO)).argmax(dim=1).cpu().numpy()

y_prueba_cnn = etq_test.numpy()
accuracy_cnn = accuracy_score(y_prueba_cnn, prediccion_cnn)
f1_cnn = f1_score(y_prueba_cnn, prediccion_cnn, average='macro')

print(f'Accuracy en prueba: {accuracy_cnn:.3f}')
print(f'F1 macro en prueba: {f1_cnn:.3f}')
print('\nReporte de clasificación:')
print(classification_report(y_prueba_cnn, prediccion_cnn, target_names=CLASES, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_prueba_cnn, prediccion_cnn, display_labels=CLASES, cmap='Greens', colorbar=False, ax=ax)
ax.set_title('Matriz de confusión: MobileNetV2 + capa final')
plt.show()

## 10. Comparación de modelos

Los dos modelos se evaluaron con el mismo conjunto de prueba. LBP + SVM utiliza exclusivamente textura en escala de grises; MobileNetV2 recibe imágenes RGB y aprovecha patrones aprendidos previamente de color, textura y forma. La **accuracy** indica la proporción total de hojas clasificadas correctamente. El **F1 macro** calcula el F1 de cada clase y luego obtiene su promedio simple, por lo que las tres clases tienen la misma importancia aunque sus cantidades no sean idénticas.

In [ ]:
df_comparacion = pd.DataFrame([
    {'Modelo': 'LBP espacial + SVM lineal', 'Tipo': 'Machine learning clásico', 'Entrada': '160 características de textura', 'Accuracy prueba': accuracy_lbp_svm, 'F1 macro prueba': f1_lbp_svm},
    {'Modelo': 'MobileNetV2 + capa final', 'Tipo': 'Transfer learning', 'Entrada': 'Imagen RGB de 224 × 224', 'Accuracy prueba': accuracy_cnn, 'F1 macro prueba': f1_cnn}
])
display(df_comparacion.style.format({'Accuracy prueba': '{:.3f}', 'F1 macro prueba': '{:.3f}'}))

fig, ax = plt.subplots(figsize=(8, 4))
posiciones = np.arange(len(df_comparacion))
ancho = 0.35
ax.bar(posiciones - ancho / 2, df_comparacion['Accuracy prueba'], ancho, label='Accuracy', color='tab:blue')
ax.bar(posiciones + ancho / 2, df_comparacion['F1 macro prueba'], ancho, label='F1 macro', color='tab:orange')
ax.set_xticks(posiciones)
ax.set_xticklabels(['LBP + SVM', 'MobileNetV2'], rotation=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Puntaje')
ax.set_title('Comparación de desempeño en el conjunto de prueba')
ax.legend()
plt.show()

## 11. Conclusiones, limitaciones y trabajo futuro

### Conclusiones

- Se implementó un enfoque clásico basado en textura local mediante LBP y SVM, y un enfoque de transferencia de aprendizaje basado en MobileNetV2.
- La comparación se realizó con las mismas clases y el mismo conjunto de prueba, mediante Accuracy y F1 macro.
- LBP permitió construir un modelo explicable y compacto, mientras que MobileNetV2 aprovechó información visual más amplia, incluyendo color, textura y forma.
- En esta ejecución, LBP + SVM alcanzó Accuracy de 0.577 y F1 macro de 0.580. MobileNetV2 alcanzó Accuracy de 0.934 y F1 macro de 0.933. Por tanto, la CNN obtuvo un desempeño notablemente mayor en el conjunto de prueba.
- El resultado sugiere que las diferencias entre las tres condiciones no dependen únicamente de microtexturas en escala de grises. La CNN aprovecha simultáneamente color, textura y patrones visuales más amplios.

### Limitaciones

- El dataset contiene solo tres condiciones visuales; el modelo no reconoce todas las enfermedades ni todas las especies de plantas.
- La clasificación corresponde a imágenes similares a las del dataset y no constituye un diagnóstico agrícola general.
- Las imágenes pueden presentar sesgos de fondo, iluminación, especie vegetal o severidad de la lesión.

### Trabajo futuro

- Evaluar imágenes tomadas en campo bajo condiciones más diversas.
- Incluir más especies y condiciones foliares.
- Aplicar segmentación para localizar la zona afectada, además de clasificar la imagen completa.
- Ajustar parcialmente las últimas capas de MobileNetV2 mediante *fine-tuning* si se dispone de GPU y más datos.

## 12. Reproducibilidad y entrega

Para publicar el código, el repositorio de GitHub debe incluir este cuadernillo, un archivo `requirements.txt`, un `README.md` con instrucciones de ejecución y la referencia al dataset. Las imágenes descargadas no es necesario subirlas a GitHub; puede indicarse en el README que se descargan desde Kaggle y se extraen en la carpeta `dataset_plant_disease/`.

En la presentación se recomienda mostrar: descripción del dataset, diagrama de metodología, ejemplos de preprocesamiento, LBP e histogramas, arquitectura MobileNetV2, métricas, matrices de confusión, comparación y conclusiones.